# 🌟 LangGraph for Kids (Super Easy Notes)

## What is LangGraph?
Imagine you are making a **sandwich** 🥪:
1. Get bread
2. Add butter
3. Add cheese
4. Done!

Each step is a small job. You do them **in order**, and you pass the sandwich from one step to the next.

**LangGraph = a tool to build these "step by step" plans for a computer (usually with AI inside).**

It is like a **board game map** 🗺️: you have places to stop, and arrows that tell you where to go next.

## 🧩 The 4 Big Words (just 4!)

| Word | Kid meaning | Sandwich example |
|---|---|---|
| **State** | A backpack 🎒 that carries information | The sandwich itself |
| **Node** | One step / one worker 👷 | "Add butter" |
| **Edge** | An arrow ➡️ saying what comes next | "After butter, add cheese" |
| **Graph** | The whole map = all nodes + edges 🗺️ | The full sandwich recipe |

Also two special spots:
- **START** 🏁 = where the game begins
- **END** 🏆 = where the game finishes

## 🎒 1. State = the backpack
The State holds the things we want to remember while walking through the map.

In Python we describe the backpack with a `TypedDict` (a dictionary that says what things it holds and their types).

In [ ]:
from typing import TypedDict

class State(TypedDict):
    message: str   # our backpack holds one thing: a message (text)

## 👷 2. Node = a worker (just a Python function!)
A node:
- **gets** the backpack (state)
- **does a small job**
- **returns** only the things it wants to change

⚠️ Important: A node does **not** return the whole backpack, only the **changes**.

In [ ]:
def say_hello(state: State):
    return {"message": "Hello"}

def add_world(state: State):
    # state["message"] is "Hello" here (from the last worker)
    return {"message": state["message"] + " World!"}

## 🗺️ 3. Build the Graph (draw the map)
Steps:
1. Make a `StateGraph` (an empty map that knows what the backpack looks like)
2. `add_node` → put workers on the map
3. `add_edge` → draw arrows between them
4. `compile()` → "lock" the map so it is ready to run

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(State)            # 1. empty map

builder.add_node("hello", say_hello)   # 2. add workers (name, function)
builder.add_node("world", add_world)

builder.add_edge(START, "hello")       # 3. arrows:  START -> hello
builder.add_edge("hello", "world")     #             hello -> world
builder.add_edge("world", END)         #             world -> END

graph = builder.compile()              # 4. ready!

## ▶️ 4. Run it!
We call `invoke()` and give the starting backpack. 🎒

In [ ]:
result = graph.invoke({"message": ""})
print(result)
# {'message': 'Hello World!'}

### What just happened? (like a movie 🎬)
```
START 🏁
  ↓
[hello]  → backpack: "Hello"
  ↓
[world]  → backpack: "Hello World!"
  ↓
END 🏆
```

## 👀 5. See the map as a picture

In [ ]:
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))

## 🔀 6. Conditional Edge = a fork in the road
Sometimes the computer must **choose** where to go, like:
> "If it is raining ☔ → take an umbrella. Otherwise → wear sunglasses 😎"

We write a small function that returns the **name of the next node**.

In [ ]:
import random

class WeatherState(TypedDict):
    weather: str
    action: str

def check_weather(state: WeatherState):
    return {"weather": random.choice(["rain", "sun"])}

def take_umbrella(state: WeatherState):
    return {"action": "Take umbrella ☔"}

def wear_sunglasses(state: WeatherState):
    return {"action": "Wear sunglasses 😎"}

# the "decider" - returns the NAME of the next node
def choose_path(state: WeatherState):
    if state["weather"] == "rain":
        return "umbrella"
    return "sunglasses"

b = StateGraph(WeatherState)
b.add_node("check", check_weather)
b.add_node("umbrella", take_umbrella)
b.add_node("sunglasses", wear_sunglasses)

b.add_edge(START, "check")
b.add_conditional_edges("check", choose_path)   # 🔀 the fork!
b.add_edge("umbrella", END)
b.add_edge("sunglasses", END)

weather_graph = b.compile()
print(weather_graph.invoke({"weather": "", "action": ""}))

## 🔁 7. Loops (go around again)
A graph can go **back** to an earlier node. Like a game: *"keep jumping until you reach 3!"*

In [ ]:
class CounterState(TypedDict):
    count: int

def add_one(state: CounterState):
    return {"count": state["count"] + 1}

def should_continue(state: CounterState):
    return END if state["count"] >= 3 else "add_one"   # stop at 3, else loop

c = StateGraph(CounterState)
c.add_node("add_one", add_one)
c.add_edge(START, "add_one")
c.add_conditional_edges("add_one", should_continue)

print(c.compile().invoke({"count": 0}))   # {'count': 3}

## 🧠 Quick Memory Cheat Sheet

| I want to... | I write... |
|---|---|
| Describe the backpack | `class State(TypedDict): ...` |
| Make a worker | a normal Python `def` that returns a dict |
| Start a map | `StateGraph(State)` |
| Add a worker | `add_node("name", function)` |
| Draw an arrow | `add_edge("from", "to")` |
| Make a fork | `add_conditional_edges("from", decider_function)` |
| Lock the map | `.compile()` |
| Run it | `.invoke({...starting state...})` |

## ❓ Why use LangGraph (and not just plain Python)?
- 🧭 You can **see** the flow as a map
- 🔀 Easy **choices** and **loops**
- 🤖 Perfect for **AI agents** that think, use tools, and try again
- 💾 Can **remember** and **pause/resume** (more advanced, learn next!)

## 🚀 What to learn next
1. Put a real **LLM** (AI brain) inside a node
2. **Tools** (let AI use a calculator or search)
3. **Memory** (checkpointers)
4. **Human-in-the-loop** (ask a person before continuing)

> 🎉 Great job! Remember: **State = backpack, Node = worker, Edge = arrow, Graph = map.**